In [1]:
#import current working directory

import os
import sys

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\research


In [2]:
# Read the sys path
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks")

In [3]:
# present working directory

%pwd


'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Handwriting_Verification_Using_Siamese_Neural_Networks\\research'

In [4]:
# move to parent directory

os.chdir("../") 

In [5]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Handwriting_Verification_Using_Siamese_Neural_Networks'

In [6]:
# test project import

import box

print(box.__version__)

7.4.1


In [7]:
# entity

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True) # dataclass immutable after creation
class DataIngestionConfig:

    root_dir: Path
    dataset_name: str
    data_source: Path
    dataset_dir: Path
    train_dir: Path
    test_dir: Path
    image_types: tuple
    annotation_dir: str


In [8]:
from Handwriting_Verification_Using_Siamese_Neural_Networks.constant import *
from Handwriting_Verification_Using_Siamese_Neural_Networks.utils.common import read_yaml,create_directories
from Handwriting_Verification_Using_Siamese_Neural_Networks.entity.config_entity import DataIngestionConfig

In [9]:
from pathlib import Path
import yaml
from box import ConfigBox

from dataclasses import dataclass

In [10]:
# Configuration Manager

class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        # Read YAML configuration
        self.config = read_yaml(config_filepath)

        # Read experiment parameters
        self.params = read_yaml(params_filepath)

        # Create artifacts root directory
        create_directories([
            PROJECT_ROOT / self.config.artifacts_root
        ])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        # Resolve dataset source path
        data_source = PROJECT_ROOT / config.data_source

        # Locate extracted CVL dataset
        if not data_source.exists():
            raise FileNotFoundError(
                f"Dataset source not found: {data_source}"
            )

        # Find the folder containing trainset and testset
        dataset_candidates = [
            path.parent
            for path in data_source.rglob("trainset")
            if path.is_dir()
            and (path.parent / "testset").is_dir()
        ]

        if len(dataset_candidates) != 1:
            raise ValueError(
                f"Expected one CVL dataset folder, "
                f"but found: {dataset_candidates}"
            )

        dataset_dir = dataset_candidates[0]

        train_dir = dataset_dir / config.train_dir
        test_dir = dataset_dir / config.test_dir

        # Create ingestion artifact directory
        root_dir = PROJECT_ROOT / config.root_dir

        create_directories([root_dir])

        data_ingestion_config = DataIngestionConfig(

            root_dir=root_dir,
            dataset_name=config.dataset_name,
            data_source=data_source,
            dataset_dir=dataset_dir,
            train_dir=train_dir,
            test_dir=test_dir,
            image_types=tuple(config.image_types),
            annotation_dir=config.annotation_dir
        )

        return data_ingestion_config

In [11]:
# import libraries for component

import os
import shutil
import re

from pathlib import Path
import json
import xml.etree.ElementTree as ET

from pathlib import Path

import pandas as pd
from PIL import Image, UnidentifiedImageError

from Handwriting_Verification_Using_Siamese_Neural_Networks.entity.config_entity import DataIngestionConfig

from Handwriting_Verification_Using_Siamese_Neural_Networks.logging import logger

from Handwriting_Verification_Using_Siamese_Neural_Networks.utils.common import get_size

In [12]:
# Components 

class DataIngestion:
    """
    CVL dataset ingestion component.

    Tasks:
    1. Dataset inventory
    2. Image inspection
    3. XML annotation inspection
    4. Save ingestion reports
    """

    def __init__(self, config: DataIngestionConfig):
        self.config = config
        self.report_dir = self.config.root_dir

    # TASK 1: VALIDATE DATASET

    def validate_dataset(self):
        required_paths = {
            "Dataset": self.config.dataset_dir,
            "Training": self.config.train_dir,
            "Testing": self.config.test_dir,
        }

        for name, path in required_paths.items():
            if not path.exists():
                raise FileNotFoundError(
                    f"{name} directory not found: {path}"
                )

            if not path.is_dir():
                raise NotADirectoryError(
                    f"{name} path is not a directory: {path}"
                )

        logger.info("CVL dataset directories validated successfully.")

    # TASK 1: DATASET INVENTORY

    def collect_file_inventory(self):
        logger.info("Starting dataset file inventory.")

        inventory = []

        dataset_splits = {
            "train": self.config.train_dir,
            "test": self.config.test_dir,
        }

        categories = [
            *self.config.image_types,
            self.config.annotation_dir,
        ]

        for split_name, split_path in dataset_splits.items():
            for category in categories:
                category_path = split_path / category

                folder_exists = category_path.is_dir()

                if folder_exists:
                    files = [
                        path
                        for path in category_path.rglob("*")
                        if path.is_file()
                    ]
                else:
                    files = []
                    logger.warning(
                        f"Category folder not found: {category_path}"
                    )

                total_size_bytes = sum(
                    path.stat().st_size for path in files
                )

                inventory.append({
                    "Split": split_name,
                    "Category": category,
                    "Folder_Exists": folder_exists,
                    "File_Count": len(files),
                    "Total_Size_MB": round(
                        total_size_bytes / (1024 ** 2), 2
                    ),
                    "Extensions": ", ".join(sorted({
                        path.suffix.lower()
                        for path in files
                        if path.suffix
                    })),
                })

        inventory_df = pd.DataFrame(inventory)

        logger.info("Dataset inventory completed.")

        return inventory_df

    # TASK 2: IMAGE INSPECTION

    def inspect_images(self):
        logger.info("Starting image inspection.")

        image_records = []

        image_extensions = {
            ".png", ".jpg", ".jpeg",
            ".bmp", ".tif", ".tiff",
        }

        dataset_splits = {
            "train": self.config.train_dir,
            "test": self.config.test_dir,
        }

        for split_name, split_path in dataset_splits.items():
            for category in self.config.image_types:
                category_path = split_path / category

                if not category_path.is_dir():
                    logger.warning(
                        f"Image category not found: {category_path}"
                    )
                    continue

                image_files = [
                    path
                    for path in category_path.rglob("*")
                    if path.is_file()
                    and path.suffix.lower() in image_extensions
                ]

                for image_path in image_files:
                    record = {
                        "Split": split_name,
                        "Category": category,
                        "Image_Name": image_path.name,
                        "Image_Path": str(image_path),
                        "Extension": image_path.suffix.lower(),
                        "Width": None,
                        "Height": None,
                        "Channels": None,
                        "Color_Mode": None,
                        "Image_Format": None,
                        "Is_Valid": False,
                        "Error": None,
                    }

                    try:
                        with Image.open(image_path) as image:
                            image.verify()

                        with Image.open(image_path) as image:
                            record["Width"] = image.width
                            record["Height"] = image.height
                            record["Channels"] = len(image.getbands())
                            record["Color_Mode"] = image.mode
                            record["Image_Format"] = image.format
                            record["Is_Valid"] = True

                    except (
                        UnidentifiedImageError,
                        OSError,
                        ValueError,
                    ) as error:
                        record["Error"] = str(error)

                        logger.warning(
                            f"Invalid image: {image_path} | {error}"
                        )

                    image_records.append(record)

        image_df = pd.DataFrame(image_records)

        valid_count = (
            int(image_df["Is_Valid"].sum())
            if not image_df.empty else 0
        )

        invalid_count = len(image_df) - valid_count

        logger.info(
            f"Image inspection completed. "
            f"Valid: {valid_count}, Invalid: {invalid_count}"
        )

        return image_df

    # TASK 3: XML PARSING

    @staticmethod
    def parse_xml_file(xml_path):
        """
        Parse XML using the original bytes first.
        If the XML declaration conflicts with the encoding,
        try supported encodings without modifying the source file.
        """

        raw_content = xml_path.read_bytes()

        try:
            return ET.fromstring(raw_content)

        except ET.ParseError:
            pass

        encodings = [
            "utf-8-sig",
            "utf-8",
            "cp1252",
            "latin-1",
        ]

        for encoding in encodings:
            try:
                decoded_content = raw_content.decode(encoding)

                decoded_content = re.sub(
                    r"""(<\?xml[^?]*encoding\s*=\s*["'])[^"']+(["'][^?]*\?>)""",
                    r"\g<1>UTF-8\2",
                    decoded_content,
                    count=1,
                    flags=re.IGNORECASE,
                )

                return ET.fromstring(
                    decoded_content.encode("utf-8")
                )

            except (
                UnicodeDecodeError,
                ET.ParseError,
                ValueError,
            ):
                continue

        raise ValueError(
            f"Unable to parse XML file: {xml_path}"
        )

    # TASK 3: XML ANNOTATION INSPECTION

    def inspect_xml_annotations(self):
        logger.info("Starting XML annotation inspection.")

        xml_records = []

        dataset_splits = {
            "train": self.config.train_dir,
            "test": self.config.test_dir,
        }

        def get_local_tag(tag):
            """
            Remove the XML namespace from a tag.

            Example:
            {namespace}Page becomes Page.
            """
            return tag.split("}")[-1]

        for split_name, split_path in dataset_splits.items():
            xml_dir = split_path / self.config.annotation_dir

            if not xml_dir.is_dir():
                logger.warning(
                    f"XML directory not found: {xml_dir}"
                )
                continue

            xml_files = sorted(xml_dir.rglob("*.xml"))

            for xml_path in xml_files:
                record = {
                    "Split": split_name,
                    "XML_File": xml_path.name,
                    "Root_Tag": None,
                    "Writer_ID": None,
                    "Writer_ID_Source": "Filename inference",
                    "Image_Reference": None,
                    "Image_Width": None,
                    "Image_Height": None,
                    "Text_Content": None,
                    "Text_Line_Count": 0,
                    "Is_Valid": False,
                    "Error": None,
                }

                try:
                    root = self.parse_xml_file(xml_path)

                    record["Root_Tag"] = get_local_tag(root.tag)

                    # Extract writer ID from the CVL filename.
                    # Example: 0001-1_attributes.xml -> 0001
                    filename_match = re.match(
                        r"^(\d+)-",
                        xml_path.stem,
                    )

                    if filename_match:
                        record["Writer_ID"] = (
                            filename_match.group(1)
                        )

                    # Extract PAGE XML metadata.
                    for element in root.iter():
                        tag = get_local_tag(element.tag)

                        if tag == "Page":
                            record["Image_Reference"] = (
                                element.attrib.get("imageFilename")
                            )

                            record["Image_Width"] = (
                                element.attrib.get("imageWidth")
                            )

                            record["Image_Height"] = (
                                element.attrib.get("imageHeight")
                            )

                    # Extract transcribed text from Unicode elements.
                    text_values = []

                    for element in root.iter():
                        tag = get_local_tag(element.tag)

                        if tag == "Unicode":
                            if (
                                element.text
                                and element.text.strip()
                            ):
                                text_values.append(
                                    element.text.strip()
                                )

                    record["Text_Content"] = " ".join(
                        text_values
                    )

                    record["Text_Line_Count"] = len(
                        text_values
                    )

                    record["Is_Valid"] = True

                except (
                    ET.ParseError,
                    OSError,
                    ValueError,
                ) as error:
                    record["Error"] = str(error)

                    logger.warning(
                        f"XML inspection failed: "
                        f"{xml_path.name} | {error}"
                    )

                xml_records.append(record)

        xml_df = pd.DataFrame(xml_records)

        valid_count = (
            int(xml_df["Is_Valid"].sum())
            if not xml_df.empty else 0
        )

        invalid_count = len(xml_df) - valid_count

        logger.info(
            f"XML inspection completed. "
            f"Valid: {valid_count}, Invalid: {invalid_count}"
        )

        return xml_df

    # TASK 4: SAVE INGESTION REPORTS

    def save_ingestion_reports(
        self,
        inventory_df,
        image_df,
        xml_df,
    ):
        self.report_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        inventory_csv = (
            self.report_dir / "dataset_inventory.csv"
        )

        image_csv = (
            self.report_dir / "image_inspection.csv"
        )

        xml_csv = (
            self.report_dir / "xml_annotation_inspection.csv"
        )

        inventory_df.to_csv(
            inventory_csv,
            index=False,
        )

        image_df.to_csv(
            image_csv,
            index=False,
        )

        xml_df.to_csv(
            xml_csv,
            index=False,
        )

        total_files = int(
            inventory_df["File_Count"].sum()
        )

        total_size_mb = round(
            float(inventory_df["Total_Size_MB"].sum()),
            2,
        )

        valid_images = (
            int(image_df["Is_Valid"].sum())
            if not image_df.empty else 0
        )

        valid_xml = (
            int(xml_df["Is_Valid"].sum())
            if not xml_df.empty else 0
        )

        summary = {
            "dataset_name": self.config.dataset_name,
            "dataset_path": str(self.config.dataset_dir),

            "dataset_inventory": {
                "total_files": total_files,
                "total_size_mb": total_size_mb,
                "categories": inventory_df.to_dict(
                    orient="records"
                ),
            },

            "image_inspection": {
                "total_images": len(image_df),
                "valid_images": valid_images,
                "invalid_images": len(image_df) - valid_images,
            },

            "xml_inspection": {
                "total_xml_files": len(xml_df),
                "valid_xml_files": valid_xml,
                "invalid_xml_files": len(xml_df) - valid_xml,
            },

            "reports": {
                "inventory_csv": inventory_csv.name,
                "image_inspection_csv": image_csv.name,
                "xml_inspection_csv": xml_csv.name,
            },
        }

        summary_json = (
            self.report_dir / "ingestion_summary.json"
        )

        with open(
            summary_json,
            "w",
            encoding="utf-8",
        ) as json_file:
            json.dump(
                summary,
                json_file,
                indent=4,
                ensure_ascii=False,
            )

        logger.info("All ingestion reports saved successfully.")

        print("\nINGESTION REPORTS SAVED")
        print(f"Inventory: {inventory_csv}")
        print(f"Image inspection: {image_csv}")
        print(f"XML inspection: {xml_csv}")
        print(f"Summary: {summary_json}")

        return summary

    # EXECUTE COMPLETE DATA INGESTION

    def initiate_data_ingestion(self):
        print("\nSTARTING WRITERLENS DATA INGESTION")

        self.validate_dataset()

        print("\n[1/4] Dataset inventory...")
        inventory_df = self.collect_file_inventory()

        print("\n[2/4] Image inspection...")
        image_df = self.inspect_images()

        print("\n[3/4] XML annotation inspection...")
        xml_df = self.inspect_xml_annotations()

        print("\n[4/4] Saving ingestion reports...")

        summary = self.save_ingestion_reports(
            inventory_df=inventory_df,
            image_df=image_df,
            xml_df=xml_df,
        )

        print("\nDATA INGESTION COMPLETED")

        return {
            "inventory": inventory_df,
            "image_inspection": image_df,
            "xml_inspection": xml_df,
            "summary": summary,
        }


In [13]:
# TASK 1: LOAD EXISTING XML INSPECTION REPORT

from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd

report_path = (
    PROJECT_ROOT
    / "artifacts"
    / "data_ingestion"
    / "xml_annotation_inspection.csv"
)

xml_df = pd.read_csv(report_path)

print("XML records:", len(xml_df))
print("Available columns:", xml_df.columns.tolist())


# TASK 2: CREATE CONFIGURATION MANAGER

# config inside DataIngestionTrainingPipeline.main()
# is local to that method. Create a new instance here.

configuration_manager = ConfigurationManager()
ingestion_config = configuration_manager.get_data_ingestion_config()

sample_xml_name = xml_df.iloc[0]["XML_File"]
sample_split = xml_df.iloc[0]["Split"]

split_dir = (
    ingestion_config.train_dir
    if sample_split == "train"
    else ingestion_config.test_dir
)

sample_xml_path = (
    split_dir
    / ingestion_config.annotation_dir
    / sample_xml_name
)

print("\nSelected XML:", sample_xml_path)
print("File exists:", sample_xml_path.exists())

if not sample_xml_path.exists():
    raise FileNotFoundError(sample_xml_path)


# TASK 3: PARSE XML

root = DataIngestion.parse_xml_file(sample_xml_path)

print("\nRoot tag:", root.tag)
print("Root attributes:", root.attrib)


# TASK 4: INSPECT XML TAG FREQUENCY

tag_counts = Counter(
    element.tag.split("}")[-1]
    for element in root.iter()
)

tag_df = pd.DataFrame(
    tag_counts.items(),
    columns=["XML_Tag", "Count"]
)

display(tag_df)


# TASK 5: DISPLAY FIRST 30 XML ELEMENTS

print("\nFirst 30 XML elements:")

for index, element in enumerate(root.iter()):
    if index >= 30:
        break

    print(f"\n{index + 1}. TAG: {element.tag.split('}')[-1]}")
    print("ATTRIBUTES:", element.attrib)
    print("TEXT:", repr(element.text))

XML records: 1604
Available columns: ['Split', 'XML_File', 'Root_Tag', 'Writer_ID', 'Writer_ID_Source', 'Image_Reference', 'Image_Width', 'Image_Height', 'Text_Content', 'Text_Line_Count', 'Is_Valid', 'Error']
[2026-11-02 01:37:01,802: INFO: common: yaml file: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\config\config.yaml loaded successfully]
[2026-11-02 01:37:01,814: INFO: common: yaml file: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\params.yaml loaded successfully]
[2026-11-02 01:37:01,816: INFO: common: created directory at: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\artifacts]
[2026-11-02 01:37:06,675: INFO: common: created directory at: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\artifacts\data_ingestion]

Selected XML: c:\Users\Greesha Vaishnavi\Desktop\

,XML_Tag,Count
0,PcGts,1
1,Metadata,1
2,Creator,1
3,Created,1
4,LastChange,1
5,Page,1
6,dkTextLines,1
7,textlines,40
8,AttrRegion,132
9,minAreaRect,132



First 30 XML elements:

1. TAG: PcGts
ATTRIBUTES: {'{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2010-03-19 http://schema.primaresearch.org/PAGE/gts/pagecontent/2010-03-19/pagecontent.xsd', 'pcGtsId': 'pc-0001-1'}
TEXT: '\n    '

2. TAG: Metadata
ATTRIBUTES: {}
TEXT: '\n        '

3. TAG: Creator
ATTRIBUTES: {}
TEXT: 'Vienna UT'

4. TAG: Created
ATTRIBUTES: {}
TEXT: '2012-09-24T14:21:58'

5. TAG: LastChange
ATTRIBUTES: {}
TEXT: '2012-09-24T14:21:58'

6. TAG: Page
ATTRIBUTES: {'imageFilename': '0001-1.tif', 'imageHeight': '3507', 'imageWidth': '2480'}
TEXT: '\n        '

7. TAG: dkTextLines
ATTRIBUTES: {'cropH': '2499', 'cropW': '2499', 'cropX': '0', 'cropY': '828'}
TEXT: '\n            '

8. TAG: textlines
ATTRIBUTES: {'xE': '2252.62', 'xS': '2050.3', 'yE': '326.451', 'yS': '353.427'}
TEXT: None

9. TAG: textlines
ATTRIBUTES: {'xE': '1664.59', 'xS': '243.438', 'yE': '404.818', 'yS': '389.334'}
TEXT: None

10. TAG: te

In [16]:
# TASK 6: INSPECT HANDWRITING LINE AND REGION ANNOTATIONS

# Select the first handwriting line
textline_element = next(
    element
    for element in root.iter()
    if element.tag.split("}")[-1] == "textlines"
)

print("TEXTLINE ELEMENT")
print("Attributes:", textline_element.attrib)
print("Text:", repr(textline_element.text))

print("\nChild elements:")

for child in textline_element:
    print(
        "Tag:", child.tag.split("}")[-1],
        "| Attributes:", child.attrib,
        "| Text:", repr(child.text)
    )


# Select the first AttrRegion
attr_region_element = next(
    element
    for element in root.iter()
    if element.tag.split("}")[-1] == "AttrRegion"
)

print("\nATTRREGION ELEMENT")
print("Attributes:", attr_region_element.attrib)

for child in attr_region_element:
    print(
        "Tag:", child.tag.split("}")[-1],
        "| Attributes:", child.attrib,
        "| Text:", repr(child.text)
    )

TEXTLINE ELEMENT
Attributes: {'xE': '2252.62', 'xS': '2050.3', 'yE': '326.451', 'yS': '353.427'}
Text: None

Child elements:

ATTRREGION ELEMENT
Attributes: {'area': '489765', 'attrType': '4', 'fontAngleRad': '0.150222', 'fontSize': '42.6257', 'fontType': '2', 'id': '0001-1', 'medianWordHeight': '1010.5'}
Tag: minAreaRect | Attributes: {} | Text: '\n                '
Tag: AttrRegion | Attributes: {'area': '200712', 'attrType': '3', 'fontAngleRad': '2.22507e-308', 'fontType': '1', 'id': '0001-1', 'medianWordHeight': '43.5'} | Text: '\n                '
Tag: AttrRegion | Attributes: {'area': '288773', 'attrType': '3', 'fontAngleRad': '2.22507e-308', 'fontType': '2', 'id': '0001-1', 'medianWordHeight': '96'} | Text: '\n                '


In [17]:
# TASK 7: INSPECT CVL IMAGE FILENAME CONVENTION

from pathlib import Path

ingestion_config = ConfigurationManager().get_data_ingestion_config()

for split_name, split_dir in [
    ("TRAIN", ingestion_config.train_dir),
    ("TEST", ingestion_config.test_dir),
]:
    print(f"\n{'=' * 60}")
    print(f"{split_name} SET")
    print(f"{'=' * 60}")

    for folder_name in ["pages", "lines", "words"]:
        folder_path = split_dir / folder_name

        files = sorted(
            path.name
            for path in folder_path.iterdir()
            if path.is_file()
        )

        print(f"\n{folder_name.upper()} — Total files: {len(files)}")
        print("First 15 filenames:")

        for filename in files[:15]:
            print(filename)

[2026-11-02 01:40:14,362: INFO: common: yaml file: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\config\config.yaml loaded successfully]
[2026-11-02 01:40:14,367: INFO: common: yaml file: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\params.yaml loaded successfully]
[2026-11-02 01:40:14,369: INFO: common: created directory at: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\artifacts]
[2026-11-02 01:40:19,901: INFO: common: created directory at: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\artifacts\data_ingestion]

TRAIN SET

PAGES — Total files: 189
First 15 filenames:
0001-1.tif
0001-2.tif
0001-3.tif
0001-4.tif
0001-6.tif
0001-7.tif
0001-8.tif
0002-1.tif
0002-2.tif
0002-3.tif
0002-4.tif
0002-6.tif
0002-7.tif
0002-8.tif
0003-1.tif

LINES — Total files: 0
First 15 filen

In [18]:
# TASK 8: VERIFY CVL DATASET DIRECTORY PATHS

print("TRAIN DIRECTORY:", ingestion_config.train_dir)
print("TEST DIRECTORY:", ingestion_config.test_dir)

print("\nDIRECTORY DETAILS")

for split_name, split_dir in [
    ("TRAIN", ingestion_config.train_dir),
    ("TEST", ingestion_config.test_dir),
]:
    print(f"\n{split_name}:")

    for folder_name in ["pages", "lines", "words", "xml"]:
        folder_path = split_dir / folder_name

        print(f"\n{folder_name}:")
        print("Path:", folder_path)
        print("Exists:", folder_path.exists())

        if folder_path.exists():
            print(
                "TIFF files:",
                len(list(folder_path.glob("*.tif")))
                + len(list(folder_path.glob("*.tiff")))
            )

TRAIN DIRECTORY: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\data\raw\CVL\cvl-database-1-1\cvl-database-1-1\trainset
TEST DIRECTORY: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\data\raw\CVL\cvl-database-1-1\cvl-database-1-1\testset

DIRECTORY DETAILS

TRAIN:

pages:
Path: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\data\raw\CVL\cvl-database-1-1\cvl-database-1-1\trainset\pages
Exists: True
TIFF files: 189

lines:
Path: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\data\raw\CVL\cvl-database-1-1\cvl-database-1-1\trainset\lines
Exists: True
TIFF files: 0

words:
Path: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\data\raw\CVL\cvl-database-1-1\cvl-database-1-1\trainset\words
Exists: True
TIFF files: 0

xml:
Path: c

In [19]:
# TASK 9: INSPECT ACTUAL CONTENTS OF LINES AND WORDS DIRECTORIES

for split_name, split_dir in [
    ("TRAIN", ingestion_config.train_dir),
    ("TEST", ingestion_config.test_dir),
]:
    for folder_name in ["lines", "words"]:
        folder_path = split_dir / folder_name

        files = [path for path in folder_path.rglob("*") if path.is_file()]

        print(f"\n{split_name} - {folder_name.upper()}")
        print("Total files:", len(files))

        print("First 10 files:")
        for file_path in files[:10]:
            print(file_path.name, "|", file_path.suffix)


TRAIN - LINES
Total files: 1624
First 10 files:
0001-1-0.tif | .tif
0001-1-1.tif | .tif
0001-1-10.tif | .tif
0001-1-2.tif | .tif
0001-1-3.tif | .tif
0001-1-4.tif | .tif
0001-1-5.tif | .tif
0001-1-6.tif | .tif
0001-1-7.tif | .tif
0001-1-8.tif | .tif

TRAIN - WORDS
Total files: 12147
First 10 files:
0001-1-0-0-Imagine.tif | .tif
0001-1-0-1-a.tif | .tif
0001-1-0-2-vast.tif | .tif
0001-1-0-3-sheet.tif | .tif
0001-1-0-4-of.tif | .tif
0001-1-0-5-paper.tif | .tif
0001-1-0-6-on.tif | .tif
0001-1-0-7-which.tif | .tif
0001-1-0-8-straight.tif | .tif
0001-1-1-0-Lines.tif | .tif

TEST - LINES
Total files: 11849
First 10 files:
0052-1-0.tif | .tif
0052-1-1.tif | .tif
0052-1-10.tif | .tif
0052-1-2.tif | .tif
0052-1-3.tif | .tif
0052-1-4.tif | .tif
0052-1-5.tif | .tif
0052-1-6.tif | .tif
0052-1-7.tif | .tif
0052-1-8.tif | .tif

TEST - WORDS
Total files: 87757
First 10 files:
0052-1-0-0-Imagine.tif | .tif
0052-1-0-1-a.tif | .tif
0052-1-0-2-vast.tif | .tif
0052-1-0-3-sheet.tif | .tif
0052-1-0-4-of.tif 

In [14]:
# Pipeline

class DataIngestionTrainingPipeline:

    def __init__(self):
        pass

    def main(self):
        
        try:
            print("\n--- STARTING DATA INGESTION PIPELINE ---")

            config = ConfigurationManager()

            data_ingestion_config = (config.get_data_ingestion_config())

            data_ingestion = DataIngestion(config=data_ingestion_config)

            results = data_ingestion.initiate_data_ingestion()

            print("\n--- DATA INGESTION PIPELINE COMPLETED ---")

            return results

        except Exception:
            logger.exception("Data Ingestion Pipeline failed.")
            raise

In [15]:
obj = DataIngestionTrainingPipeline()
obj.main()


--- STARTING DATA INGESTION PIPELINE ---
[2026-11-02 01:37:06,756: INFO: common: yaml file: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\config\config.yaml loaded successfully]
[2026-11-02 01:37:06,760: INFO: common: yaml file: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\params.yaml loaded successfully]
[2026-11-02 01:37:06,762: INFO: common: created directory at: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\artifacts]
[2026-11-02 01:37:07,198: INFO: common: created directory at: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Handwriting_Verification_Using_Siamese_Neural_Networks\artifacts\data_ingestion]

STARTING WRITERLENS DATA INGESTION
[2026-11-02 01:37:07,198: INFO: 4168527755: CVL dataset directories validated successfully.]

[1/4] Dataset inventory...
[2026-11-02 01:37:07,198: INFO: 4168527755: Starting dataset 

KeyboardInterrupt: 